In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))


In [3]:
import importlib
import src.oos
importlib.reload(src.oos)

oos_results = src.oos.run()
oos_results.round(4)

A_tfidf        {'val': (1430, 70), 'test_in': (2800, 70), 'test_idoos': (280, 70), 'test_oodoos': (1000, 70)}
B_minilm       {'val': (1430, 70), 'test_in': (2800, 70), 'test_idoos': (280, 70), 'test_oodoos': (1000, 70)}
C_distilbert   {'val': (1430, 70), 'test_in': (2800, 70), 'test_idoos': (280, 70), 'test_oodoos': (1000, 70)}

=== deployed score: energy, target coverage 95% ===
       track  threshold  coverage_test_in  rejected_idoos  rejected_oodoos  acc_all  acc_on_accepted
     A_tfidf     4.8869            0.9568          0.2714           0.8310   0.8871           0.9026
    B_minilm     6.0098            0.9596          0.4071           0.9690   0.9214           0.9352
C_distilbert     4.5827            0.9643          0.4714           0.9440   0.9207           0.9367


,track,score,auroc_idoos,auroc_oodoos,threshold,coverage_test_in,rejected_idoos,rejected_oodoos,acc_on_accepted,acc_all
0,A_tfidf,msp,0.8618,0.9586,0.1988,0.9575,0.3536,0.778,0.9097,0.8871
1,A_tfidf,margin,0.8459,0.9319,0.0491,0.9496,0.2929,0.591,0.9165,0.8871
2,A_tfidf,neg_entropy,0.8584,0.9668,-3.3872,0.9596,0.2714,0.828,0.9032,0.8871
3,A_tfidf,max_logit,0.8490,0.9663,3.3100,0.9593,0.3000,0.815,0.9066,0.8871
4,A_tfidf,energy,0.8283,0.9676,4.8869,0.9568,0.2714,0.831,0.9026,0.8871
5,B_minilm,msp,0.9198,0.9765,0.3188,0.9539,0.5107,0.878,0.9442,0.9214
6,B_minilm,margin,0.9037,0.9534,0.0880,0.9593,0.3786,0.681,0.9415,0.9214
7,B_minilm,neg_entropy,0.9241,0.9870,-2.4411,0.9543,0.5464,0.945,0.9401,0.9214
8,B_minilm,max_logit,0.9178,0.9893,5.0594,0.9546,0.4857,0.955,0.9401,0.9214
9,B_minilm,energy,0.9014,0.9922,6.0098,0.9596,0.4071,0.969,0.9352,0.9214


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.oos import collect_logits, confidence_scores
from src.splits import load_label_map

logits, splits = collect_logits()
SCORE = "energy"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
cov_grid = np.linspace(0.50, 0.999, 200)

for track, per_split in logits.items():
    s_in = confidence_scores(per_split["test_in"])[SCORE]
    s_id = confidence_scores(per_split["test_idoos"])[SCORE]
    s_ood = confidence_scores(per_split["test_oodoos"])[SCORE]

    ths = np.quantile(s_in, 1.0 - cov_grid)
    axes[0].plot(cov_grid, [(s_id < t).mean() for t in ths], label=track)
    axes[1].plot(cov_grid, [(s_ood < t).mean() for t in ths], label=track)

for ax, title in zip(axes, ["ID-OOS (held-out banking intents)", "OOD-OOS (non-banking)"]):
    ax.axvline(0.95, ls="--", c="grey", lw=1)
    ax.set_xlabel("coverage on in-scope test")
    ax.set_title(title)
axes[0].set_ylabel("fraction of unknowns rejected")
axes[0].legend()
plt.tight_layout()
fig.savefig(config.FIGURES_DIR / "coverage_vs_rejection.png", dpi=150)
plt.show()

inv = {v: k for k, v in load_label_map().items()}

y_true = splits["test_in"]["label"].to_numpy()
y_pred = logits["B_minilm"]["test_in"].argmax(axis=1)
conf = pd.DataFrame({"true": [inv[i] for i in y_true], "predicted": [inv[i] for i in y_pred]})
conf = conf[conf["true"] != conf["predicted"]]
print("top in-scope confusions (Track B):")
print(conf.value_counts().head(12).to_string())

id_pred = logits["B_minilm"]["test_idoos"].argmax(axis=1)
routed = pd.DataFrame({
    "held_out_intent": splits["test_idoos"]["intent"],
    "routed_to": [inv[i] for i in id_pred],
})
print("\nwhere held-out intents get routed (Track B):")
print(routed.value_counts().head(12).to_string())

A_tfidf        {'val': (1430, 70), 'test_in': (2800, 70), 'test_idoos': (280, 70), 'test_oodoos': (1000, 70)}
B_minilm       {'val': (1430, 70), 'test_in': (2800, 70), 'test_idoos': (280, 70), 'test_oodoos': (1000, 70)}
C_distilbert   {'val': (1430, 70), 'test_in': (2800, 70), 'test_idoos': (280, 70), 'test_oodoos': (1000, 70)}


<Figure size 1200x450 with 2 Axes>

top in-scope confusions (Track B):
true                                     predicted                     
declined_transfer                        declined_card_payment             6
card_arrival                             card_delivery_estimate            6
get_disposable_virtual_card              getting_virtual_card              5
pending_transfer                         transfer_timing                   5
card_payment_not_recognised              compromised_card                  5
beneficiary_not_allowed                  failed_transfer                   4
fiat_currency_support                    exchange_via_app                  4
balance_not_updated_after_bank_transfer  transfer_timing                   4
beneficiary_not_allowed                  transfer_into_account             3
declined_transfer                        failed_transfer                   3
pending_cash_withdrawal                  cash_withdrawal_not_recognised    3
why_verify_identity                      verif

In [5]:
import time, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from src import config

d = config.MODELS_DIR / "track_c" / "distilbert_intent"
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(d)
mdl = AutoModelForSequenceClassification.from_pretrained(d)
mdl.eval()
print(f"loaded in {time.perf_counter()-t0:.1f}s | labels: {mdl.config.num_labels}")

with torch.no_grad():
    out = mdl(**tok("my card was declined at the atm", return_tensors="pt"))
print("logits:", out.logits.shape, "| argmax:", out.logits.argmax(-1).item())


loaded in 0.9s | labels: 70
logits: torch.Size([1, 70]) | argmax: 24
